In [2]:

import numpy as np
import pandas as pd

np.random.seed(42)

n = 200

# Choose cluster sizes (sum must equal n)
sizes = [70, 65, 65]  # you can change these proportions if you like

# Cluster distribution parameters (overlapping)
# Spending: lognormal where median = exp(mu); sigma controls skew
cluster_spending_median = [150, 500, 280]   # medians for clusters (overlap)
cluster_spending_sigma  = [0.3, 0.2, 0.25]  # variability

# Visits: Poisson lambda (overlap)
cluster_visits_lambda = [3, 12, 8]

# Annual Income (k$): normal mean and std (overlap)
cluster_income_mean = [80, 150, 195]
cluster_income_std  = [20, 15, 18]

records = []

for cluster_id, size in enumerate(sizes):
    # Spending: compute mu for lognormal from desired median
    mu = np.log(cluster_spending_median[cluster_id])
    sigma = cluster_spending_sigma[cluster_id]
    spending = np.random.lognormal(mean=mu, sigma=sigma, size=size)
    # Clip spending to [100, 2000]
    spending = np.clip(spending, 100, 2000)

    # Visits: Poisson, then clip to [2,20]
    visits = np.random.poisson(lam=cluster_visits_lambda[cluster_id], size=size)
    visits = np.clip(visits, 2, 20)

    # Annual Income (k$): normal, clip to [60,250]
    income = np.random.normal(loc=cluster_income_mean[cluster_id],
                              scale=cluster_income_std[cluster_id],
                              size=size)
    income = np.clip(income, 60, 250)

    for s, v, inc in zip(spending, visits, income):
        records.append({
            'Spending': float(s),
            'Visits': int(v),
            'Annual Income (k$)': float(inc),
            'Cluster': int(cluster_id)
        })



In [3]:
# Build DataFrame, shuffle, assign CustomerID
df = pd.DataFrame(records).sample(frac=1, random_state=42).reset_index(drop=True)
df.insert(0, 'CustomerID', range(1, len(df) + 1))

df.to_csv('customer_data_clustering.csv', index=False)